# 消息类型

在 LangChain 中，所有对话都通过消息（Message）对象传递。理解各种消息类型的用途是编写 Agent 的基础。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## 四种核心消息类型

| 类型 | 角色 | 说明 | 典型内容 |
| --- | --- | --- | --- |
| HumanMessage | 用户 | 用户发送的消息 | "今天天气怎么样？" |
| AIMessage | AI 助手 | 模型回复，可能包含 tool_calls | "今天杭州晴天，25°C" |
| SystemMessage | 系统 | 系统指令，定义 AI 角色和行为规则 | "你是一个专业的天气助手" |
| ToolMessage | 工具 | 工具执行后的返回结果 | "晴，25°C，湿度 60%" |

## HumanMessage——用户消息

代表用户发送给 AI 的消息，是对话的起点。

In [5]:
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

# 创建一条用户消息
msg = HumanMessage(content="特朗普是谁？")
print(f"类型: {msg.type}")
print(f"内容: {msg.content}")

# 创建消息列表（多轮对话历史）
messages = [
    HumanMessage(content="你好"),
    HumanMessage(content="特朗普是谁？"),
]

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
response = model.invoke(messages)
print(f"\n模型回复: {response.content}")

类型: human
内容: 特朗普是谁？

模型回复: 你好！特朗普（唐纳德·特朗普，Donald Trump）是美国的一位政治人物、商人和媒体名人。他曾在2017年至2021年担任美国第45任总统，代表共和党。在从政之前，他以房地产开发商和真人秀节目《学徒》的主持人而闻名。他的总统任期以“美国优先”政策、减税、移民限制以及对贸易政策的调整而著称，同时也伴随着较大争议。如果你有更具体的问题，我很乐意提供更多信息！


### HumanMessage 快捷创建方式

使用元组或字典作为快捷方式，在 Agent 内部会自动转换为 HumanMessage。

In [6]:
from langchain.messages import HumanMessage

# 四种方式等价
msg1 = HumanMessage(content="你好")       # 标准构造
msg2 = ("user", "你好")                   # 元组快捷方式
msg3 = {"role": "user", "content": "你好"}  # 字典快捷方式

print(f"msg1 类型: {type(msg1).__name__}")
print(f"msg2 类型: {type(msg2).__name__}")
print(f"msg3 类型: {type(msg3).__name__}")

msg1 类型: HumanMessage
msg2 类型: tuple
msg3 类型: dict


## AIMessage——AI 回复

代表模型的回复，可能包含 tool_calls（工具调用请求）。

In [7]:
from langchain.messages import AIMessage

# 普通 AI 回复（无工具调用）
ai_msg = AIMessage(content="特朗普（唐纳德·特朗普，Donald Trump）是美国的一位政治人物、商人和媒体名人。")

# 包含工具调用的 AI 回复
ai_with_tools = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"city": "杭州"},
            "id": "call_abc123",
            "type": "tool_call",
        }
    ]
)

print("=== 普通 AI 消息 ===")
print(f"content: {ai_msg.content}")
print(f"tool_calls: {ai_msg.tool_calls}")
print("\n=== 含工具调用的 AI 消息 ===")
print(f"tool_calls: {ai_with_tools.tool_calls}")

=== 普通 AI 消息 ===
content: 特朗普（唐纳德·特朗普，Donald Trump）是美国的一位政治人物、商人和媒体名人。
tool_calls: []

=== 含工具调用的 AI 消息 ===
tool_calls: [{'name': 'get_weather', 'args': {'city': '杭州'}, 'id': 'call_abc123', 'type': 'tool_call'}]


In [8]:
from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek:deepseek-v4-flash")
response = model.invoke("介绍特朗普")

print(f"内容: {response.content}")
print(f"模型名: {response.response_metadata.get('model_name')}")
print(f"完成原因: {response.response_metadata.get('finish_reason')}")

if response.usage_metadata:
    print(f"输入 tokens: {response.usage_metadata.get('input_tokens')}")
    print(f"输出 tokens: {response.usage_metadata.get('output_tokens')}")

内容: 唐纳德·特朗普（Donald Trump）是美国第45任总统（2017年1月20日至2021年1月20日），也是美国历史上首位没有担任过任何公职或军职的总统。他出生于1946年6月14日，是一位房地产商人、电视节目主持人，曾任特朗普集团董事长兼总裁。

特朗普的政治立场以民粹主义、民族主义和保护主义为标志。他的竞选口号“让美国再次伟大”强调美国优先、限制移民、重新谈判贸易协定、打击“政治正确”以及通过减税和放松监管刺激经济。他在任期间推行了大幅减税法案、改革司法体系、退出多项国际协议（如《巴黎气候协定》和伊朗核协议），并推动美中贸易摩擦。此外，他因“通俄门”调查、两次弹劾案以及应对新冠疫情的方式引发巨大争议。

2020年大选失利后，特朗普未承认败选，并指控选举舞弊。2021年1月6日，他的部分支持者冲击美国国会大厦，导致特朗普因“煽动叛乱”被第二次弹劾，但最终被判无罪。目前，他仍是共和党内具有重要影响力的政治人物，并已宣布参选2024年总统选举。

特朗普的言行充满争议，支持者认为他打破了传统政治规则、维护美国利益，而批评者则指责其言论分裂社会、削弱民主制度。
模型名: deepseek-v4-flash
完成原因: stop
输入 tokens: 6
输出 tokens: 392


## SystemMessage——系统指令

设定 AI 的行为、角色和约束。放在消息列表最前面。

In [9]:
from langchain.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0.7)

# 有系统指令的回复
messages_with_system = [
    SystemMessage(content="你是一个小红书风格的博主，回复要活泼、使用 emoji、带话题标签"),
    HumanMessage(content="介绍特朗普")
]
response = model.invoke(messages_with_system)
print(f"系统指令效果: {response.content}")

系统指令效果: 家人们谁懂啊！今天要介绍一个自带热搜体质的男人——川建国同志！👴🏻🇺🇸

他，是房地产大亨转型综艺顶流，再跨界当上美国总统的“斜杠老年”！📺💰  
他，金发像被炸过，握手像拔河，推特治国的第一人！📱💥  
他，自称“懂王”，没人比他更懂病毒、关税、甚至汉堡怎么夹肉！🍔🧠  

经典语录：“你被解雇了！”（附带甩手手势）🤚  
经典造型：西装+红领带+惊悚肤色对比⚡  

现在虽然不在白宫了，但依然活跃在“让美国再次伟大”的直播间（和法庭门口）😂  
姐妹们，咱就是说，这老头儿的人生比美剧还带劲！🔥  

#懂王传奇 #川建国 #美国总统 #喜剧人设 #吃瓜日常


## ToolMessage——工具返回结果

必须与对应的 tool_call 关联，`tool_call_id` 需精确匹配。

In [13]:
from langchain.messages import HumanMessage, AIMessage, ToolMessage

# 模拟一轮完整的工具调用对话
# 注意：DeepSeek V4 Flash 的 thinking mode 要求回传 reasoning_content，
# 因此手动构建带 tool_calls 的 AIMessage 后无法直接 invoke。
# 这里仅展示 ToolMessage 的构造方式，实际运行请使用 Agent。
messages = [
    HumanMessage(content="杭州天气怎么样？"),
    AIMessage(
        content="",
        tool_calls=[
            {"name": "get_weather", "args": {"city": "杭州"},
             "id": "call_abc", "type": "tool_call"}
        ]
    ),
    ToolMessage(
        content="晴，25°C，湿度 60%",
        tool_call_id="call_abc",
        name="get_weather",
    ),
]

print(f"ToolMessage 构造完成，共 {len(messages)} 条消息")
print(f"最后一条: type={messages[-1].type}, tool_call_id={messages[-1].tool_call_id}")

ToolMessage 构造完成，共 3 条消息
最后一条: type=tool, tool_call_id=call_abc


> `tool_call_id` 必须与 `AIMessage` 中 `tool_call` 的 `id` 精确匹配，否则模型可能忽略工具结果。

## AIMessageChunk——流式输出的消息片段

使用 `stream()` 时，每个到达的片段是 `AIMessageChunk`。

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek:deepseek-v4-flash")

# for chunk in model.stream("用一句话介绍菜鸟教程 RUNOOB"):
#     print(chunk.content, end="", flush=True)

print("取消注释上方代码测试流式输出")

## 消息类型速查表

| 消息类型 | type 属性 | role 属性 | 关键字段 |
| --- | --- | --- | --- |
| HumanMessage | human | user | content |
| AIMessage | ai | assistant | content, tool_calls, usage_metadata |
| AIMessageChunk | ai | assistant | content（增量） |
| SystemMessage | system | system | content |
| ToolMessage | tool | tool | content, tool_call_id, name |

## ContentBlock——结构化消息内容

消息内容可以是多个 ContentBlock 组成的列表，用于混合文本和图片。

In [ ]:
from langchain.messages import HumanMessage

# content 可以是纯字符串（简单场景）
simple_msg = HumanMessage(content="你好")

# content 也可以是 ContentBlock 列表（复杂场景：图文混合）
complex_msg = HumanMessage(content=[
    {"type": "text", "text": "这张图片里是什么？"},
    {"type": "image_url", "image_url": {"url": "https://example.com/photo.jpg"}},
])

print(f"简单消息 content 类型: {type(simple_msg.content).__name__}")
print(f"复杂消息 content 类型: {type(complex_msg.content).__name__}")
print(f"内容块数量: {len(complex_msg.content)}")

> 纯文本场景直接传字符串即可。需要混合文本和图片时，才手动构建 ContentBlock 列表。

## ToolCall——工具调用

`AIMessage` 中的 `tool_calls` 字段是一个 `ToolCall` 列表。

In [ ]:
from langchain.messages import AIMessage

# 手动构建 ToolCall
tool_call = {
    "name": "get_weather",
    "args": {"city": "杭州"},
    "id": "call_abc123",
    "type": "tool_call",
}

ai_message = AIMessage(content="", tool_calls=[tool_call])
print(f"工具名称: {ai_message.tool_calls[0]['name']}")
print(f"调用参数: {ai_message.tool_calls[0]['args']}")

## trim_messages()——裁剪消息历史

对话太长时，智能裁剪消息历史以适应模型上下文窗口。

In [ ]:
from langchain.messages import (
    HumanMessage, AIMessage, SystemMessage, trim_messages
)

# 模拟长对话
messages = [
    SystemMessage(content="你是菜鸟教程 RUNOOB 的 AI 助手"),
    HumanMessage(content="Python 怎么入门？"),
    AIMessage(content="Python 入门可以从基础知识开始..."),
    HumanMessage(content="有推荐的 IDE 吗？"),
    AIMessage(content="推荐 VS Code 或 PyCharm..."),
    HumanMessage(content="如何安装第三方库？"),
    AIMessage(content="使用 pip install 命令..."),
    HumanMessage(content="NumPy 是什么？"),
    AIMessage(content="NumPy 是一个科学计算库..."),
]

# 裁剪消息（保留 system + 最近的对话）
trimmed = trim_messages(
    messages,
    max_tokens=1000,
    strategy="last",
    token_counter=len,  # 简单计数
    include_system=True,
    start_on="human",
)

print(f"裁剪前: {len(messages)} 条消息")
print(f"裁剪后: {len(trimmed)} 条消息")
for msg in trimmed:
    snippet = msg.content[:50] if isinstance(msg.content, str) else str(msg.content)[:50]
    print(f"  [{msg.type}] {snippet}")

## RemoveMessage——删除特定消息

从消息历史中按 ID 删除特定消息，用于内容清洗或重新生成。

In [ ]:
from langchain.messages import RemoveMessage

# 创建 RemoveMessage 指定要删除的消息 ID
removal = RemoveMessage(id="msg_3")
print(f"要删除的消息 ID: {removal.id}")
print(f"类型: {removal.type}")

> `RemoveMessage` 配合 `add_messages` reducer 使用，在 middleware 或钩子中返回可动态清理消息历史。

## 消息属性通用方法

所有消息类型继承自 `BaseMessage`，共享以下通用方法。

In [ ]:
from langchain.messages import HumanMessage

msg = HumanMessage(content="你好，菜鸟教程")

print(f"content: {msg.content}")
print(f"type: {msg.type}")
print(f"id: {msg.id}")
print(f"text: {msg.text}")
print(f"美化输出:\n{msg.pretty_repr()}")

**术语：**

- **HumanMessage**：用户消息，对话的起点
- **AIMessage**：AI 回复，可能包含 tool_calls
- **SystemMessage**：系统指令，定义 AI 角色和行为
- **ToolMessage**：工具执行结果，必须关联 tool_call_id
- **AIMessageChunk**：流式输出的消息片段
- **ContentBlock**：结构化消息内容（文本、图片等）
- **trim_messages()**：裁剪消息历史以适应上下文窗口